<div align="center">
  <a href="https://pageindex.ai/developer">
    <img src="https://github.com/user-attachments/assets/bae02956-6c4e-4a0b-adea-257b0be4aaa1" width="100%" alt="PageIndex — Vectorless, Reasoning-based RAG">
  </a>
  <p>
    <a href="https://pageindex.ai/developer">🌐 Website</a> &nbsp; · &nbsp;
    <a href="https://developer.pageindex.ai/">☁️ Cloud</a> &nbsp; · &nbsp;
    <a href="https://docs.pageindex.ai/">📖 Docs</a> &nbsp; · &nbsp;
    <a href="https://github.com/VectifyAI/PageIndex">⭐ GitHub</a> &nbsp; · &nbsp;
    <a href="https://pageindex.ai/contact">✉️ Contact</a>
  </p>
</div>

---

# PageIndex Document Metadata

**Tag your documents when you upload them, then ask by tag.**

Tags carry what a file does not say about itself: which team owns it, what kind of document it is, which year it belongs to. This notebook demonstrates how to attach **your own metadata** to documents on PageIndex Cloud and have the chat agent filter and count documents by it.

We declare a folder's metadata fields, `type`, `team`, and `year`, then upload four documents into it, each tagged with its values. Because the fields are declared, the agent can search the folder by them: it counts documents per team, or picks one out by its tags, instead of matching file names.

1. **Declare the fields** — Give a folder its fields with `create_metadata_schema()`.
2. **Tag on upload** — Pass a `metadata` dict to `submit_document()`.
3. **Ask by tag** — Count and find documents by their tags in chat.

**Before you start:** Have your [PageIndex API key](https://developer.pageindex.ai/api-keys) and OpenAI API key ready. This demo creates a folder in your PageIndex Cloud library and uploads four small PDFs into it.


## 01 · Set up

Install the latest PageIndex SDK. Requests downloads the demo documents.


In [ ]:
!pip install -U pageindex requests

### Configure your client

Set your API keys and choose your chat `MODEL`. The client indexes documents on PageIndex Cloud, and your chosen model answers with PageIndex Cloud's retrieval tools.


In [2]:
import os
from pageindex import PageIndexClient

os.environ["PAGEINDEX_API_KEY"] = "Your PageIndex API key here"
os.environ["OPENAI_API_KEY"] = "Your OpenAI API key here"

MODEL = "gpt-5.6-luna"  # gpt-5.6-sol / gpt-5.6-terra / gpt-5.6-luna

client = PageIndexClient(
    index="cloud",
    chat=MODEL,
)

## 02 · Declare the fields

Create a top-level folder and declare its fields. Each field has a type, `string`, `number`, or `boolean`, and a description that tells the agent what it holds. Subfolders share the fields, and a folder's fields are set once. A folder left by an earlier run is reused.


In [3]:
from pageindex import PageIndexAPIError

FOLDER = "Metadata Demo"
FIELDS = {
    "type": {"type": "string", "description": "Document kind: earnings_release, regulation, or paper"},
    "team": {"type": "string", "description": "Team that owns the document"},
    "year": {"type": "number", "description": "Year the document was published"},
}

try:
    folder_id = client.get_folder_id(FOLDER)  # created on an earlier run
except PageIndexAPIError:
    folder_id = client.create_folder(FOLDER)["folder"]["id"]
    client.create_metadata_schema(folder_id, FIELDS)
print(f"{FOLDER}: {folder_id}")

Metadata Demo: cmvi17aah00040ap9cfez1efj


## 03 · Tag documents on upload

Download four PDFs from the PageIndex repository and submit each one into the folder with its tags. `team` is something no file states about itself; only the tag knows it. `wait=True` returns once the document is ready. A document already in your library from an earlier run is reused rather than uploaded again.


In [4]:
import requests

BASE_URL = "https://raw.githubusercontent.com/VectifyAI/PageIndex/main/examples/documents/"
DOCS = {
    "q1-fy25-earnings.pdf": {"type": "earnings_release", "team": "finance", "year": 2025},
    "Regulation Best Interest_Interpretive release.pdf": {"type": "regulation", "team": "legal", "year": 2019},
    "earthmover.pdf": {"type": "paper", "team": "research", "year": 2013},
    "attention-residuals.pdf": {"type": "paper", "team": "research", "year": 2026},
}

doc_ids = {}
for file_name, metadata in DOCS.items():
    try:
        doc_ids[file_name] = client.get_document_id(file_name)  # already in your library
        print(f"{file_name}: {doc_ids[file_name]} (already uploaded)")
        continue
    except PageIndexAPIError:
        pass

    response = requests.get(BASE_URL + file_name, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)

    doc_ids[file_name] = client.submit_document(
        file_name, wait=True, folder_id=folder_id, metadata=metadata,
    )["doc_id"]
    print(f"{file_name}: {doc_ids[file_name]}")

q1-fy25-earnings.pdf: pi-cmvi1etih00050bp9uh6viy63
Regulation Best Interest_Interpretive release.pdf: pi-cmvi1fysj00040apbh630pbwe
earthmover.pdf: pi-cmvi1grw300020dpbm3eknt7l
attention-residuals.pdf: pi-cmvi1hl1900080cpb8cgysmub


The tags come back under `metadata` from `get_document()`, and on every document from `list_documents()`. Metadata is set at submission; to change it, delete the document and submit it again.


In [5]:
for file_name, doc_id in doc_ids.items():
    print(f"{file_name}: {client.get_document(doc_id)['metadata']}")

q1-fy25-earnings.pdf: {'team': 'finance', 'type': 'earnings_release', 'year': 2025}
Regulation Best Interest_Interpretive release.pdf: {'team': 'legal', 'type': 'regulation', 'year': 2019}
earthmover.pdf: {'team': 'research', 'type': 'paper', 'year': 2013}
attention-residuals.pdf: {'team': 'research', 'type': 'paper', 'year': 2026}


## 04 · Ask by tag

Ask how many documents each team owns. No document says which team owns it, so the answer has to come from the tags: the agent reads the folder's fields with `get_metadata_schema`, then counts by `team` with `search_documents(mode="metadata")`. The output streams the answer alongside retrieval tool calls.


In [6]:
messages = [{"role": "user", "content": "How many documents does each team own?"}]

response = None
for event in client.chat(messages, folder_id=folder_id, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] get_metadata_schema({"folder_id":"cmvi17aah00040ap9cfez1efj"})

[Tool call] search_documents({"mode":"metadata","folder_id":"cmvi17aah00040ap9cfez1efj","recursive":true,"limit":50,"metadata":{"group_by":["team"],"aggregates":{"document_count":{"$count":{}}},"sort":{"team":1}}})
Document ownership by team in **Metadata Demo**:

- Finance: 1 document
- Legal: 1 document
- Research: 2 documents

Total: 4 documents.

Tags also filter. Ask which documents are from 2019 or later: the agent can filter the folder by `year` with a range, as in the run below, rather than guessing dates from the documents' text. With only four documents it may read the years straight off a folder listing instead.


In [7]:
messages = [{"role": "user", "content": "Which documents are from 2019 or later? List each with its team and year."}]

response = None
for event in client.chat(messages, folder_id=folder_id, stream=True, protocol="responses"):
    if event["type"] == "response.output_text.delta":
        print(event["delta"], end="", flush=True)
    elif event["type"] == "response.output_item.done":
        item = event["item"]
        if item.get("type") == "function_call":
            print(f"\n[Tool call] {item['name']}({item.get('arguments', '')})", flush=True)
    elif event["type"] in {"response.completed", "response.incomplete", "response.failed"}:
        response = event["response"]


[Tool call] get_metadata_schema({"folder_id":"cmvi17aah00040ap9cfez1efj"})

[Tool call] search_documents({"mode":"metadata","folder_id":"cmvi17aah00040ap9cfez1efj","recursive":true,"metadata":{"metadata_filter":{"year":{"$gte":2019}},"sort":{"year":1}},"limit":50})
| Document | Team | Year |
|---|---|---:|
| Regulation Best Interest_Interpretive release.pdf | legal | 2019 |
| q1-fy25-earnings.pdf | finance | 2025 |
| attention-residuals.pdf | research | 2026 |

Metadata is a flat dict of `str`, `int`, `float`, or `bool` values (numbers must be finite, so no `NaN` or `Infinity`), with keys that start with a letter and contain only letters, digits, and underscores. Its serialized size is at most 8 KB. In a folder with declared fields, a document may only use those fields, each with its declared type. PageIndex Cloud rejects metadata that breaks a rule with `PageIndexAPIError`. See the [metadata documentation](https://docs.pageindex.ai/sdk/documents#metadata) for details.


## 05 · Review usage & estimated cost (optional)

Inspect the token usage for the last answer and estimate its model cost using the rate table below. Declaring fields, tagging documents, and reading the tags back do not require model calls. This estimate does not include PageIndex Cloud charges.


In [8]:
import json

# Standard USD per 1M tokens: (input, cached input, output).
# Checked 2026-09-17: https://developers.openai.com/api/docs/models/{model}
PRICES = {
    "gpt-5.6-sol": (4.0, 0.4, 20.0),
    "gpt-5.6-terra": (2.0, 0.2, 12.0),
    "gpt-5.6-luna": (0.2, 0.02, 1.2),
}
input_rate, cached_rate, output_rate = PRICES[MODEL]

usage = response.get("usage") if response else None
print("\n\nUsage:", json.dumps(usage, ensure_ascii=False, indent=2))
if response and response.get("status") != "completed":
    print("Response did not complete:", response.get("error") or response.get("incomplete_details"))

if usage:
    details = usage.get("input_tokens_details") or {}
    cached = details.get("cached_tokens", 0) or 0
    written = details.get("cache_write_tokens", 0) or 0
    regular = usage["input_tokens"] - cached - written
    cost = (regular * input_rate + cached * cached_rate + written * input_rate * 1.25
            + usage["output_tokens"] * output_rate) / 1_000_000
    print(f"Estimated model cost at base rates: ${cost:.6f} USD")
    if usage["input_tokens"] > 272_000:
        print("Usage is aggregated across model calls; the estimate excludes possible long-context surcharges.")
else:
    print("No usage returned; unable to estimate cost.")



Usage: {
  "input_tokens": 17903,
  "input_tokens_details": {
    "cached_tokens": 11301,
    "cache_write_tokens": 6593
  },
  "output_tokens": 233,
  "output_tokens_details": {
    "reasoning_tokens": 64
  },
  "total_tokens": 18136,
  "request_usage_entries": [
    {
      "input_tokens": 5439,
      "input_tokens_details": {
        "cached_tokens": 0,
        "cache_write_tokens": 5436
      },
      "output_tokens": 81,
      "output_tokens_details": {
        "reasoning_tokens": 45
      },
      "total_tokens": 5520
    },
    {
      "input_tokens": 5868,
      "input_tokens_details": {
        "cached_tokens": 5436,
        "cache_write_tokens": 429
      },
      "output_tokens": 87,
      "output_tokens_details": {
        "reasoning_tokens": 19
      },
      "total_tokens": 5955
    },
    {
      "input_tokens": 6596,
      "input_tokens_details": {
        "cached_tokens": 5865,
        "cache_write_tokens": 728
      },
      "output_tokens": 65,
      "output_tokens

---

<div align="center">
  <br>
  <p>
    <a href="https://docs.pageindex.ai/"><b>Upload your documents</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://docs.pageindex.ai/sdk/agents"><b>Integrate with your agent</b></a>
    &nbsp;&nbsp; · &nbsp;&nbsp;
    <a href="https://github.com/VectifyAI/PageIndex"><b>Star PageIndex on GitHub</b></a>
  </p>
  <br>
  <a href="https://pageindex.ai">
    <img src="https://github.com/user-attachments/assets/e1c677f2-b590-4ffc-859d-e8cc7d794bdf" width="144" height="40" alt="PageIndex">
  </a>
</div>
